## 1. Setup

In [ ]:
import gc, hashlib, json, math, os, random, time
from collections import Counter

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/ethio-fakenews"
OUT_DIR = f"{DRIVE_DIR}/verdict-model"
CHECKPOINT_DIR = f"{DRIVE_DIR}/checkpoint"
STATE_PATH = f"{CHECKPOINT_DIR}/state.json"

MODEL_NAME = "Davlan/afro-xlmr-large-61L"

EPOCHS = 5
PATIENCE = 2
MAX_LEN = 256
MAX_TOKENS_PER_BATCH = 3072   
ACCUM_STEPS = 2
LEARNING_RATE = 1.5e-5
WARMUP_FRACTION = 0.06
FREEZE_EMBEDDINGS = True

RESUME = True
SAVE_EVERY_MINUTES = 30      
RESUME_WARMUP_STEPS = 100

MAX_TRAIN_EXAMPLES = 100000
MAX_VAL_EXAMPLES = 2000
SEED = 42

LABELS = ["supported", "contradicted", "not_enough_evidence"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"

if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

if DEVICE == "cpu":
    print("WARNING: no GPU found. Training will be extremely slow. Change the runtime type to T4 GPU.")

print("Training on:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Load the data

In [ ]:
def norm(text):
    return " ".join(str(text).split())


with open(f"{DRIVE_DIR}/gemini_examples.jsonl", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]
print(f"Gemini examples in the file: {len(rows)}")

audit = pd.read_csv(f"{DRIVE_DIR}/gemini_audit.csv", encoding="utf-8-sig")
if "your_check" in audit.columns:
    audit = audit[audit["your_check"].astype(str).str.strip().str.lower() != "wrong"]
audit["label"] = audit["gemini_label"].astype(str).str.strip().str.lower()
test_rows = audit[audit["label"].isin(LABELS)][["language", "claim", "evidence", "label"]].to_dict("records")

test_evidence = {norm(r["evidence"]) for r in test_rows}
test_articles = {r["article"] for r in rows if norm(r["evidence"]) in test_evidence}
rows = [r for r in rows if r["article"] not in test_articles]
print(f"after leaving out the test articles: {len(rows)}")
if not rows:
    raise SystemExit("There are no examples to train on. Check gemini_examples.jsonl in your Drive folder.")

val_rows = [r for r in rows if int(r["article"], 16) % 10 == 1]
train_rows = [r for r in rows if int(r["article"], 16) % 10 != 1]
if len(train_rows) > MAX_TRAIN_EXAMPLES:
    train_rows = random.sample(train_rows, MAX_TRAIN_EXAMPLES)
if len(val_rows) > MAX_VAL_EXAMPLES:
    val_rows = random.sample(val_rows, MAX_VAL_EXAMPLES)

fingerprint = hashlib.md5("".join(f'{r["claim"]}|{r["label"]}' for r in train_rows).encode("utf-8")).hexdigest()

print(f"using: train {len(train_rows)} | validation {len(val_rows)} | test {len(test_rows)}")
print("train labels:   ", dict(Counter(r["label"] for r in train_rows)))
print("train languages:", dict(Counter(r["language"] for r in train_rows)))
print("test languages: ", dict(Counter(r["language"] for r in test_rows)))

## 3. Helpers for fast batches and for resuming

In [ ]:
POOL_SIZE = 2000


def make_batches(lengths, seed, max_tokens):
    rng = random.Random(seed)
    order = list(range(len(lengths)))
    rng.shuffle(order)
    batches = []
    for start in range(0, len(order), POOL_SIZE):
        pool = sorted(order[start:start + POOL_SIZE], key=lambda i: lengths[i])
        batch, longest = [], 0
        for i in pool:
            if batch and max(longest, lengths[i]) * (len(batch) + 1) > max_tokens:
                batches.append(batch)
                batch, longest = [], 0
            batch.append(i)
            longest = max(longest, lengths[i])
        if batch:
            batches.append(batch)
    rng.shuffle(batches)
    return batches


def make_lr_factor(start_step, total_steps, warmup_steps, resumed, rewarm_steps):
    def factor(local_step):
        step = start_step + local_step
        if step < warmup_steps:
            value = step / max(1, warmup_steps)
        else:
            value = max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
        if resumed:
            value *= min(1.0, (local_step + 1) / rewarm_steps)
        return value
    return factor


def run_epochs(state, epoch_batches, do_batch, validate, save_best, save_checkpoint, log,
               epochs, accum, patience, save_every_seconds, clock=time.time):
    last_save = clock()
    while state["epoch"] <= epochs and not state["finished"]:
        epoch = state["epoch"]
        batches = epoch_batches(epoch)

        for step in range(state["batch_in_epoch"], len(batches)):
            done = step + 1
            at_boundary = done % accum == 0 or done == len(batches)
            do_batch(batches[step], at_boundary)

            if at_boundary:
                state["global_step"] += 1
                state["batch_in_epoch"] = done

                if done < len(batches) and clock() - last_save >= save_every_seconds:
                    save_checkpoint(state)
                    last_save = clock()

            log(epoch, done, len(batches))

        val_f1 = validate()
        if val_f1 > state["best_f1"]:
            state["best_f1"] = val_f1
            state["bad_epochs"] = 0
            save_best()
        else:
            state["bad_epochs"] += 1

        state["epoch"] += 1
        state["batch_in_epoch"] = 0
        state["finished"] = state["epoch"] > epochs or state["bad_epochs"] >= patience
        save_checkpoint(state)
        last_save = clock()

    return state


## 4. Turn text into model input

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def encode(items, chunk=5000):
    encoded = []
    for start in range(0, len(items), chunk):
        part = items[start:start + chunk]
        batch = tokenizer(
            [r["claim"] for r in part],
            [r["evidence"] for r in part],
            truncation="only_second",
            max_length=MAX_LEN
        )
        encoded.extend(
            {"ids": ids, "label": LABEL2ID[r["label"]]}
            for ids, r in zip(batch["input_ids"], part)
        )
        if len(items) > chunk:
            print(f"  tokenized {len(encoded)}/{len(items)}")
    return encoded


train_enc, val_enc, test_enc = encode(train_rows), encode(val_rows), encode(test_rows)
train_lengths = [len(e["ids"]) for e in train_enc]
print(
    "input length in tokens: median", int(np.median(train_lengths)),
    "| 95th percentile", int(np.percentile(train_lengths, 95)),
    "| MAX_LEN", MAX_LEN
)


def collate(indices, encoded):
    """Pad a batch using PyTorch's optimized pad_sequence implementation."""
    sequences = [torch.tensor(encoded[i]["ids"], dtype=torch.long) for i in indices]
    ids = torch.nn.utils.rnn.pad_sequence(
        sequences, batch_first=True, padding_value=tokenizer.pad_token_id
    )
    mask = ids.ne(tokenizer.pad_token_id).long()
    labels = torch.tensor(
        [encoded[i]["label"] for i in indices], dtype=torch.long
    )
    return ids, mask, labels


@torch.no_grad()
def predict(model, encoded):
    model.eval()
    order = sorted(range(len(encoded)), key=lambda i: len(encoded[i]["ids"]))
    batches, batch, longest = [], [], 0

    for i in order:
        n = len(encoded[i]["ids"])
        if batch and max(longest, n) * (len(batch) + 1) > 2 * MAX_TOKENS_PER_BATCH:
            batches.append(batch)
            batch, longest = [], 0
        batch.append(i)
        longest = max(longest, n)

    if batch:
        batches.append(batch)

    predictions = np.zeros(len(encoded), dtype=int)

    for indices in batches:
        ids, mask, _ = collate(indices, encoded)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(
                input_ids=ids.to(DEVICE, non_blocking=True),
                attention_mask=mask.to(DEVICE, non_blocking=True)
            ).logits
        predictions[indices] = logits.argmax(dim=-1).cpu().numpy()

    return predictions


def report(items, predictions, title):
    truth = np.array([LABEL2ID[r["label"]] for r in items])
    print(f"\n=== {title} ({len(items)} examples) ===")
    print(
        f"accuracy {accuracy_score(truth, predictions):.3f} | "
        f"macro-F1 {f1_score(truth, predictions, average='macro', labels=[0, 1, 2], zero_division=0):.3f}"
    )

    languages = np.array([r["language"] for r in items])
    for language in sorted(set(languages)):
        mask = languages == language
        f1 = f1_score(
            truth[mask], predictions[mask],
            average="macro", labels=[0, 1, 2], zero_division=0
        )
        print(
            f"  {language}: {mask.sum():4d} examples | "
            f"accuracy {accuracy_score(truth[mask], predictions[mask]):.3f} | "
            f"macro-F1 {f1:.3f}"
        )

    print(
        "confusion matrix (rows = true, columns = predicted, order:",
        ", ".join(LABELS), ")"
    )
    print(confusion_matrix(truth, predictions, labels=[0, 1, 2]))
    return f1_score(
        truth, predictions, average="macro",
        labels=[0, 1, 2], zero_division=0
    )


## 5. Full checkpoint and resume


In [ ]:
import shutil

CHECKPOINT_VERSION = 2


def save_weights(model, folder):
    """Save model weights in half precision to reduce Drive usage."""
    os.makedirs(folder, exist_ok=True)
    half = {
        k: (v.half() if v.is_floating_point() else v)
        for k, v in model.state_dict().items()
    }
    model.save_pretrained(folder, state_dict=half)


def save_full_checkpoint(state):
    """Save a complete training state to the inactive rotating slot.

    state.json is written last. Therefore, if Colab dies during a save,
    the previous slot remains the valid resume point.
    """
    folder = "b" if state.get("slot") == "a" else "a"
    checkpoint_path = f"{CHECKPOINT_DIR}/{folder}"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(checkpoint_path, exist_ok=True)

    save_weights(model, checkpoint_path)

    tokenizer.save_pretrained(checkpoint_path)

    torch.save(
        {
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict() if USE_AMP else None,
        },
        f"{checkpoint_path}/training_state.pt"
    )

    rng_payload = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        rng_payload["cuda"] = torch.cuda.get_rng_state_all()

    torch.save(rng_payload, f"{checkpoint_path}/rng_state.pt")

    state["slot"] = folder
    state["fingerprint"] = fingerprint
    state["checkpoint_version"] = CHECKPOINT_VERSION

    with open(STATE_PATH + ".tmp", "w", encoding="utf-8") as f:
        json.dump(state, f)
    os.replace(STATE_PATH + ".tmp", STATE_PATH)

    print(
        f"  saved FULL resume checkpoint -> {folder} "
        f"(epoch {state['epoch']}, batch {state['batch_in_epoch']}, "
        f"global step {state['global_step']})"
    )


def save_best():
    save_weights(model, OUT_DIR)
    tokenizer.save_pretrained(OUT_DIR)
    print(f"  saved the best model so far to {OUT_DIR}")


state = None
resumed = False

if RESUME and os.path.exists(STATE_PATH):
    with open(STATE_PATH, encoding="utf-8") as f:
        candidate = json.load(f)

    slot = candidate.get("slot")
    checkpoint_path = f"{CHECKPOINT_DIR}/{slot}" if slot in ("a", "b") else ""

    required = [
        f"{checkpoint_path}/config.json",
        f"{checkpoint_path}/training_state.pt",
        f"{checkpoint_path}/rng_state.pt",
    ]

    if (
        candidate.get("checkpoint_version") == CHECKPOINT_VERSION
        and candidate.get("fingerprint") == fingerprint
        and all(os.path.exists(p) for p in required)
    ):
        state = candidate
        resumed = True
    elif candidate.get("fingerprint") not in (None, fingerprint):
        raise SystemExit(
            "The training data is not the same as when the resume point was saved. "
            "Put the original gemini_examples.jsonl back, or set RESUME = False to start again."
        )
    else:
        print("An old/incomplete checkpoint was found; starting a fresh run.")

if not resumed:
    state = {
        "epoch": 1,
        "batch_in_epoch": 0,
        "global_step": 0,
        "best_f1": -1.0,
        "bad_epochs": 0,
        "finished": False,
        "slot": None,
        "checkpoint_version": CHECKPOINT_VERSION,
    }
else:
    print(
        f"Resuming from epoch {state['epoch']}, "
        f"batch {state['batch_in_epoch']} "
        f"(best score so far {state['best_f1']:.3f})"
    )


## 6. Model and optimizer



In [ ]:
start_from = f"{CHECKPOINT_DIR}/{state['slot']}" if resumed else MODEL_NAME
model = AutoModelForSequenceClassification.from_pretrained(
    start_from, num_labels=len(LABELS)
).to(DEVICE)

if FREEZE_EMBEDDINGS:
    for parameter in model.base_model.embeddings.word_embeddings.parameters():
        parameter.requires_grad = False

counts = Counter(r["label"] for r in train_rows)
weights = torch.tensor(
    [len(train_rows) / (len(LABELS) * max(counts[label], 1)) for label in LABELS],
    dtype=torch.float,
    device=DEVICE
)
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=0.01
)

batches_in_epoch_1 = len(
    make_batches(train_lengths, SEED + 1, MAX_TOKENS_PER_BATCH)
)
total_steps = int(
    math.ceil(batches_in_epoch_1 / ACCUM_STEPS) * EPOCHS * 1.02
)
warmup_steps = int(WARMUP_FRACTION * total_steps)

lr_factor = make_lr_factor(
    state["global_step"],
    total_steps,
    warmup_steps,
    resumed,
    RESUME_WARMUP_STEPS
)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

if resumed:
    checkpoint_path = f"{CHECKPOINT_DIR}/{state['slot']}"

    training_state = torch.load(
        f"{checkpoint_path}/training_state.pt",
        map_location=DEVICE
    )
    optimizer.load_state_dict(training_state["optimizer"])
    scheduler.load_state_dict(training_state["scheduler"])
    if USE_AMP and training_state.get("scaler") is not None:
        scaler.load_state_dict(training_state["scaler"])

    rng_state = torch.load(
        f"{checkpoint_path}/rng_state.pt",
        map_location="cpu",
        weights_only=False
    )
    random.setstate(rng_state["python"])
    np.random.set_state(rng_state["numpy"])
    torch.set_rng_state(rng_state["torch"])
    if torch.cuda.is_available() and "cuda" in rng_state:
        torch.cuda.set_rng_state_all(rng_state["cuda"])

print(
    f"{batches_in_epoch_1} batches per epoch | "
    f"{total_steps} update steps planned in total"
)


## 7. Train

In [ ]:
progress = {"processed": 0, "loss_sum": 0.0, "started": time.time()}


def do_batch(indices, at_boundary):
    model.train()
    ids, mask, labels = collate(indices, train_enc)

    ids = ids.to(DEVICE, non_blocking=True)
    mask = mask.to(DEVICE, non_blocking=True)
    labels = labels.to(DEVICE, non_blocking=True)

    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
        logits = model(input_ids=ids, attention_mask=mask).logits

    loss = loss_fn(logits.float(), labels) / ACCUM_STEPS
    scaler.scale(loss).backward()

    progress["processed"] += 1
    progress["loss_sum"] += loss.item() * ACCUM_STEPS

    if at_boundary:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)


def log(epoch, done, total):
    if done % 250 == 0 or done == total:
        seconds_per_batch = (
            (time.time() - progress["started"]) / max(1, progress["processed"])
        )
        batches_left = (total - done) + (EPOCHS - epoch) * total
        print(
            f"epoch {epoch}/{EPOCHS} batch {done}/{total} | "
            f"loss {progress['loss_sum'] / max(1, progress['processed']):.4f} | "
            f"about {seconds_per_batch * batches_left / 3600:.1f} hours left"
        )


def validate():
    return report(
        val_rows,
        predict(model, val_enc),
        f"validation after epoch {state['epoch']}"
    )


optimizer.zero_grad(set_to_none=True)

if state["finished"]:
    print("Training is already finished. Going straight to the final test.")
else:
    run_epochs(
        state,
        lambda epoch: make_batches(
            train_lengths, SEED + epoch, MAX_TOKENS_PER_BATCH
        ),
        do_batch,
        validate,
        save_best,
        save_full_checkpoint,
        log,
        epochs=EPOCHS,
        accum=ACCUM_STEPS,
        patience=PATIENCE,
        save_every_seconds=SAVE_EVERY_MINUTES * 60
    )

    if state["bad_epochs"] >= PATIENCE:
        print(f"Stopped early: no improvement for {PATIENCE} epochs.")
    print("Training finished.")


## 8. Final test

In [ ]:
del model, optimizer, scheduler, scaler
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

if os.path.exists(f"{OUT_DIR}/config.json"):
    final_dir = OUT_DIR
else:
    with open(STATE_PATH, encoding="utf-8") as f:
        final_dir = f"{CHECKPOINT_DIR}/{json.load(f)['slot']}"

best = AutoModelForSequenceClassification.from_pretrained(final_dir).to(DEVICE)
report(test_rows, predict(best, test_enc), "test on gemini_audit.csv")
